# SVM_Hyperparameter_Tuning

1. Split data 70% train / 15% validation / 15% test
2. Randomly sample 10,000 rows from train and 2,500 rows from validation
3. Drop categories with fewer than 5 samples (cannot be used in 5-fold CV)
4. Randomized search with 5-fold stratified cross-validation
5. Save only the best hyperparameters (JSON) and the 5-fold results (CSV)

## [1] Imports, paths and data loading

In [1]:
from pathlib import Path
import json
import time
import pandas as pd
import numpy as np

from scipy.stats import loguniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

input_file = (
    Path.home() / "Documents" / "AIE410_Project"
    / "Data_Preprocessed" / "Data_output.parquet"
)
output_dir = Path.home() / "Documents" / "AIE410_Project" / "SVM_TFIDF" / "Hyperparameter_Tuning_Results"
output_dir.mkdir(parents=True, exist_ok=True)

hyperparams_file = output_dir / "best_hyperparameters.json"
cv_results_file = output_dir / "hyperparameter_5fold_results.csv"

if not input_file.exists():
    raise FileNotFoundError(f"File not found: {input_file}")

df = pd.read_parquet(input_file)

required_columns = ["JASCCode", "Discrepancy"]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (1765824, 76)


,OperatorControlNumber,DifficultyDate,SubmissionDate,OperatorDesignator,SubmitterDesignator,SubmitterTypeCode,ReceivingRegionCode,ReceivingDistrictOffice,SDRType,JASCCode,...,ButtLineFromSide,ButtlineTo,ButtlineToSide,WaterLineFrom,WaterLineTo,CrackLength,NumberOfCracks,CorrosionLevel,StructuralOther,Discrepancy
0,95UALD10001,01/03/1995,2000-03-26T17:11:31.060-05:00,UALA,<NA>,A,WP,29,A,3397,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,EEL LIGHT ON RT AISLE B ZONE INOPERATIVE. REP...
1,9500042202,01/03/1995,2000-03-26T17:11:31.060-05:00,NWAA,NWAA,A,GL,01,A,2741,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"DURING 5 DEGREE FLAP TAKEOFF AT 20 KNOTS, CENT..."
2,95UALD10002,01/03/1995,2000-03-26T17:11:31.060-05:00,UALA,<NA>,A,WP,29,A,3397,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,EEL STRIP LIGHT IN B ZONE LEFT AND RIGHT SIDE ...
3,95UAL757001,01/04/1995,2000-03-26T17:11:31.060-05:00,UALA,UALA,A,WP,29,A,2560,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"DURING OVERHAUL OF THE RIGHT OFF-WING SLIDE, T..."
4,CKSA94353,01/03/1995,2000-03-26T17:11:31.060-05:00,CKSA,CKSA,A,GL,23,A,7250,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,"KMCI - FLT 59901 - ON CLIMB-OUT, NR 3 ENGINE H..."


## [2] NLTK / HTTPS setup

In [2]:
import os
import ssl
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

import re
import nltk
from functools import lru_cache
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(pkg, quiet=True)

## [3] Text cleaning and preprocessing
**Keep this cell identical in both notebooks** so the cleaned data (and therefore the split) is identical.

In [3]:
# Use ALL data here: the 100k / 25k sampling for tuning happens AFTER the split
SAMPLE_SIZE = None
RANDOM_STATE = 42
NORMALIZATION = "lemma"   # "lemma", "stem", or None

STOP_WORDS = set(stopwords.words("english")) - {"no", "not", "nor", "without"}

_norm = {
    "lemma": WordNetLemmatizer().lemmatize,
    "stem": PorterStemmer().stem,
}.get(NORMALIZATION, lambda t: t)
_norm = lru_cache(maxsize=None)(_norm)


def preprocess_text(text):
    tokens = re.sub(r"[\W_]+", " ", str(text).lower()).split()
    return " ".join(_norm(t) for t in tokens if t not in STOP_WORDS)


data = df[["Discrepancy", "JASCCode"]].copy()
data["Discrepancy"] = data["Discrepancy"].astype("string").str.strip()
data["JASCCode"] = pd.to_numeric(data["JASCCode"], errors="coerce")
data = data.dropna()
data = data[
    (data["Discrepancy"] != "")
    & (data["JASCCode"] >= 0)
    & (data["JASCCode"] % 1 == 0)
]

# JASC category = first two digits of the code (e.g. 3210 -> "32")
data["JASC_Category"] = data["JASCCode"].astype("int64").astype(str).str[:2]

if SAMPLE_SIZE and len(data) > SAMPLE_SIZE:
    data = data.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)

data["Clean_Text"] = data["Discrepancy"].map(preprocess_text)

data = (
    data[data["Clean_Text"] != ""]
    .drop_duplicates(["Clean_Text", "JASC_Category"])
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

X, y = data["Clean_Text"], data["JASC_Category"]
print(f"Samples: {len(data):,} | Categories: {y.nunique():,}")
display(y.value_counts().sort_index().rename("SampleCount").to_frame())
display(data[["Discrepancy", "Clean_Text", "JASC_Category"]].head())

Samples: 1,706,220 | Categories: 51


,SampleCount
JASC_Category,
11,1093
12,27
14,978
18,10
21,47023
22,7051
23,7770
24,19086
25,74148


,Discrepancy,Clean_Text,JASC_Category
0,AIRCRAFT WAS NOT GROUNDED: 4R DOOR SLIDE BUSTL...,aircraft not grounded 4r door slide bustlers l...,25
1,ENGINE WAS NOT MAKING FULL RPM. PROP GOV WAS ...,engine not making full rpm prop gov replaced g...,85
2,CORROSION ON LT SIDE SEAT RAIL FS 573-FS 661. ...,corrosion lt side seat rail f 573 f 661 c chec...,53
3,"PNX - FLT 515 - BLOCK TURNBACK, ON RIGHT RUNUP...",pnx flt 515 block turnback right runup nr 1 en...,73
4,NR 6 SLAT ACTUATOR INBD SUPPORT ANGLE IS CRACK...,nr 6 slat actuator inbd support angle cracked ...,57


## [4] Split: 70% train / 15% validation / 15% test
Fixed `RANDOM_STATE` + identical preprocessing => identical split in both notebooks.

In [4]:
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

class_counts = y.value_counts()
if len(data) < 20:
    raise ValueError("Not enough samples for a 70/15/15 split.")
if y.nunique() < 2:
    raise ValueError("At least 2 distinct JASC categories are required.")

# Stratify only if every category is large enough to appear in all three splits
can_stratify = class_counts.min() >= 5

# Step 1: 70% train vs 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=VAL_FRAC + TEST_FRAC,
    random_state=RANDOM_STATE,
    stratify=y if can_stratify else None,
)
# Step 2: split the 30% temp half/half -> 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
    random_state=RANDOM_STATE,
    stratify=y_temp if can_stratify else None,
)

print(f"Stratified split: {can_stratify}")
print(f"Train: {len(X_train):,} ({len(X_train)/len(data):.1%})")
print(f"Val:   {len(X_val):,} ({len(X_val)/len(data):.1%})")
print(f"Test:  {len(X_test):,} ({len(X_test)/len(data):.1%})")

Stratified split: True
Train: 1,194,354 (70.0%)
Val:   255,933 (15.0%)
Test:  255,933 (15.0%)


## [5] Randomly sample the tuning data (100,000 train + 25,000 validation)

In [5]:
N_TUNE_TRAIN = 150000
N_TUNE_VAL = 50000

# If a split has fewer rows than requested, use everything it has
tune_train_idx = X_train.sample(n=min(N_TUNE_TRAIN, len(X_train)), random_state=RANDOM_STATE).index
tune_val_idx = X_val.sample(n=min(N_TUNE_VAL, len(X_val)), random_state=RANDOM_STATE).index

# Combine the two samples into one pool for 5-fold cross-validation
X_tune = pd.concat([X_train.loc[tune_train_idx], X_val.loc[tune_val_idx]])
y_tune = pd.concat([y_train.loc[tune_train_idx], y_val.loc[tune_val_idx]])

print(f"Tuning train sample: {len(tune_train_idx):,}")
print(f"Tuning val sample:   {len(tune_val_idx):,}")
print(f"Combined tuning set: {len(X_tune):,}")

# 5-fold CV needs at least 5 samples per category (ignore categories with fewer)
N_FOLDS = 5
tune_counts = y_tune.value_counts()
dropped = tune_counts[tune_counts < N_FOLDS]
keep = tune_counts[tune_counts >= N_FOLDS].index

if len(dropped):
    print(f"\nIgnoring {len(dropped)} categories with fewer than {N_FOLDS} samples:")
    print(dropped.to_string())

mask = y_tune.isin(keep)
X_tune, y_tune = X_tune[mask], y_tune[mask]
print(f"\nFinal tuning set: {len(X_tune):,} samples | {y_tune.nunique()} categories")

Tuning train sample: 150,000
Tuning val sample:   50,000
Combined tuning set: 200,000

Ignoring 2 categories with fewer than 5 samples:
JASC_Category
12    4
45    3

Final tuning set: 199,993 samples | 48 categories


## [6] Randomized search with 5-fold cross-validation

In [6]:
N_ITER = 50
SCORING = "f1_macro"
N_JOBS = 1

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        dual="auto",
        max_iter=5000,
        random_state=RANDOM_STATE
    )),
])

param_distributions = {
    # TF-IDF
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 3],
    "tfidf__max_df": [0.85, 0.95, 1.0],

    # Linear SVM
    "classifier__C": loguniform(0.05, 10),
    "classifier__class_weight": [None, "balanced"],
    "classifier__tol": loguniform(1e-5, 1e-3),
    "classifier__loss": ["hinge", "squared_hinge"],
    #"classifier__fit_intercept": [True, False],
}

cv = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    scoring=SCORING,
    cv=cv,
    n_jobs=N_JOBS,
    pre_dispatch=1,
    refit=False,
    random_state=RANDOM_STATE,
    verbose=2
)

start = time.time()
search.fit(X_tune, y_tune)

print(f"\nSearch finished in {(time.time() - start) / 60:.1f} minutes")
print(f"Best CV {SCORING}: {search.best_score_:.4f}")
print("Best params:", search.best_params_)


Fitting 5 folds for each of 50 candidates, totalling 250 fits
[CV] END classifier__C=0.36374585440139057, classifier__class_weight=None, classifier__loss=hinge, classifier__tol=0.000291063591313307, tfidf__max_df=0.85, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  36.8s
[CV] END classifier__C=0.36374585440139057, classifier__class_weight=None, classifier__loss=hinge, classifier__tol=0.000291063591313307, tfidf__max_df=0.85, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  37.3s
[CV] END classifier__C=0.36374585440139057, classifier__class_weight=None, classifier__loss=hinge, classifier__tol=0.000291063591313307, tfidf__max_df=0.85, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  33.1s
[CV] END classifier__C=0.36374585440139057, classifier__class_weight=None, classifier__loss=hinge, classifier__tol=0.000291063591313307, tfidf__max_df=0.85, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  30.4s
[CV] END classifier__C=0.36374585440139057, classifier__cl

/Users/nileshsaumyadasa/Documents/AIE410_Project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END classifier__C=1.187834103462719, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=1.5030900645056805e-05, tfidf__max_df=1.0, tfidf__min_df=3, tfidf__ngram_range=(1, 1); total time=  45.8s
[CV] END classifier__C=1.187834103462719, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=1.5030900645056805e-05, tfidf__max_df=1.0, tfidf__min_df=3, tfidf__ngram_range=(1, 1); total time=  46.5s
[CV] END classifier__C=1.187834103462719, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=1.5030900645056805e-05, tfidf__max_df=1.0, tfidf__min_df=3, tfidf__ngram_range=(1, 1); total time=  41.3s
[CV] END classifier__C=1.187834103462719, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=1.5030900645056805e-05, tfidf__max_df=1.0, tfidf__min_df=3, tfidf__ngram_range=(1, 1); total time=  38.1s


/Users/nileshsaumyadasa/Documents/AIE410_Project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END classifier__C=1.187834103462719, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=1.5030900645056805e-05, tfidf__max_df=1.0, tfidf__min_df=3, tfidf__ngram_range=(1, 1); total time=  38.0s
[CV] END classifier__C=8.14058589987916, classifier__class_weight=None, classifier__loss=squared_hinge, classifier__tol=0.0003123490722120775, tfidf__max_df=1.0, tfidf__min_df=2, tfidf__ngram_range=(1, 2); total time= 1.5min
[CV] END classifier__C=8.14058589987916, classifier__class_weight=None, classifier__loss=squared_hinge, classifier__tol=0.0003123490722120775, tfidf__max_df=1.0, tfidf__min_df=2, tfidf__ngram_range=(1, 2); total time= 1.5min
[CV] END classifier__C=8.14058589987916, classifier__class_weight=None, classifier__loss=squared_hinge, classifier__tol=0.0003123490722120775, tfidf__max_df=1.0, tfidf__min_df=2, tfidf__ngram_range=(1, 2); total time= 1.6min
[CV] END classifier__C=8.14058589987916, classifier__class_weight=None, classifier__loss=squared_hinge

/Users/nileshsaumyadasa/Documents/AIE410_Project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


... (13 outputs left out)

/Users/nileshsaumyadasa/Documents/AIE410_Project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END classifier__C=2.164444278857321, classifier__class_weight=None, classifier__loss=hinge, classifier__tol=4.981177005235498e-05, tfidf__max_df=1.0, tfidf__min_df=3, tfidf__ngram_range=(1, 1); total time= 1.1min
[CV] END classifier__C=2.164444278857321, classifier__class_weight=None, classifier__loss=hinge, classifier__tol=4.981177005235498e-05, tfidf__max_df=1.0, tfidf__min_df=3, tfidf__ngram_range=(1, 1); total time= 1.0min
[CV] END classifier__C=1.9446366425291812, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=0.0007472397689332936, tfidf__max_df=0.85, tfidf__min_df=2, tfidf__ngram_range=(1, 2); total time= 1.4min
[CV] END classifier__C=1.9446366425291812, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=0.0007472397689332936, tfidf__max_df=0.85, tfidf__min_df=2, tfidf__ngram_range=(1, 2); total time= 1.3min
[CV] END classifier__C=1.9446366425291812, classifier__class_weight=balanced, classifier__loss=hinge, classifier__to

/Users/nileshsaumyadasa/Documents/AIE410_Project/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[CV] END classifier__C=2.458806474360191, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=0.00012103081218396826, tfidf__max_df=0.95, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  44.3s
[CV] END classifier__C=2.458806474360191, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=0.00012103081218396826, tfidf__max_df=0.95, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  51.0s
[CV] END classifier__C=2.458806474360191, classifier__class_weight=balanced, classifier__loss=hinge, classifier__tol=0.00012103081218396826, tfidf__max_df=0.95, tfidf__min_df=2, tfidf__ngram_range=(1, 1); total time=  51.2s
[CV] END classifier__C=1.6504907355754492, classifier__class_weight=balanced, classifier__loss=squared_hinge, classifier__tol=0.00012894414393074176, tfidf__max_df=0.85, tfidf__min_df=3, tfidf__ngram_range=(1, 1); total time= 1.3min
[CV] END classifier__C=1.6504907355754492, classifier__class_weight=balanced, classifier__loss=

## [7] Save results and best hyperparameters

In [7]:
results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
results.to_csv(cv_results_file, index=False)
display(results[["rank_test_score", "mean_test_score", "std_test_score", "params"]].head(10))

# Convert numpy types / tuples to plain JSON-friendly values
def to_jsonable(v):
    if isinstance(v, tuple):
        return list(v)
    if isinstance(v, np.generic):
        return v.item()
    return v

best = {k: to_jsonable(v) for k, v in search.best_params_.items()}

payload = {
    "best_cv_score": float(search.best_score_),
    "scoring": SCORING,
    "n_folds": N_FOLDS,
    "n_iter": N_ITER,
    "random_state": RANDOM_STATE,
    "tuning_samples": int(len(X_tune)),
    "best_params": best,
}

# Final JSON file with the best hyperparameters
with open(hyperparams_file, "w") as f:
    json.dump(payload, f, indent=4)

print("Saved:")
for p in [hyperparams_file, cv_results_file]:
    print(" -", p)
print("\n" + hyperparams_file.read_text())

,rank_test_score,mean_test_score,std_test_score,params
42,1,0.734407,0.008289,"{'classifier__C': 1.5007352386789095, 'classif..."
43,2,0.733346,0.008254,"{'classifier__C': 1.0842947084433703, 'classif..."
46,3,0.732910,0.008857,"{'classifier__C': 0.2557567631476557, 'classif..."
34,4,0.732309,0.011274,"{'classifier__C': 1.4346331833767147, 'classif..."
33,5,0.731955,0.009805,"{'classifier__C': 2.828001167996658, 'classifi..."
6,6,0.731412,0.007734,"{'classifier__C': 1.250112023666978, 'classifi..."
9,7,0.729199,0.009624,"{'classifier__C': 1.6728371068484111, 'classif..."
22,8,0.726131,0.008508,"{'classifier__C': 6.127843763160042, 'classifi..."
19,9,0.724916,0.007768,"{'classifier__C': 8.608127341766876, 'classifi..."
44,10,0.724639,0.010106,"{'classifier__C': 0.11721795873532405, 'classi..."


Saved:
 - /Users/nileshsaumyadasa/Documents/AIE410_Project/SVM_TFIDF/Hyperparameter_Tuning_Results/best_hyperparameters.json
 - /Users/nileshsaumyadasa/Documents/AIE410_Project/SVM_TFIDF/Hyperparameter_Tuning_Results/hyperparameter_5fold_results.csv

{
    "best_cv_score": 0.7344070795923016,
    "scoring": "f1_macro",
    "n_folds": 5,
    "n_iter": 50,
    "random_state": 42,
    "tuning_samples": 199993,
    "best_params": {
        "classifier__C": 1.5007352386789095,
        "classifier__class_weight": null,
        "classifier__loss": "hinge",
        "classifier__tol": 2.1050258247483826e-05,
        "tfidf__max_df": 0.95,
        "tfidf__min_df": 3,
        "tfidf__ngram_range": [
            1,
            2
        ]
    }
}
